In [1]:
!pip install -U langchain-neo4j
!pip install requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.6/48.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.3/325.3 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.0/234.0 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.9/183.9 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 22.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.12.0 which is incompatible.


In [2]:
!pip install fastapi uvicorn langchain langchain-openai langchain-community langchain-neo4j neo4j
!pip install requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 182.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 87.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.19
    Uninstalling langchain-core-1.2.19:
      Successfully uninstalled langchain-core-1.2.19
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which i

In [1]:
import nest_asyncio
import threading
import uvicorn
import time
import re
from typing import List, Dict
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
from google.colab import userdata, output
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# 최신 패키지 임포트
from langchain_neo4j import Neo4jGraph, Neo4jVector

# Colab 비동기 환경 패치
nest_asyncio.apply()
app = FastAPI()

# --- 1. 설정 및 리트리버 준비 ---
NEO4J_URI = ''
NEO4J_USERNAME = ''
NEO4J_PASSWORD = ''
NEO4J_DATABASE = ''
OPENAI_API_KEY = userdata.get('')

graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE)
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
llm = ChatOpenAI(model="gpt-4o", temperature=0, openai_api_key=OPENAI_API_KEY)

vector_index = Neo4jVector.from_existing_index(
    embeddings, url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE,
    index_name="entity_vector_index", text_node_property="id"
)

# --- 2. SCM 분석 및 뉴스 요약 로직 ---
news_summary_prompt = ChatPromptTemplate.from_template(
    "다음 뉴스 원문을 SCM(공급망) 리스크 관점에서 1~2문장으로 핵심만 요약해줘. 관련 링크가 있다면 포함해.\n\n원문: {text}"
)
news_summary_chain = news_summary_prompt | llm | StrOutputParser()

def graph_retriever(question: str) -> str:
    question_vector = embeddings.embed_query(question)

    # 📌 MATCH를 OPTIONAL MATCH로 수정하여 이웃이 없는 단일 노드도 안전하게 추출
    query = """
    CALL db.index.vector.queryNodes('entity_vector_index', 5, $embedding)
    YIELD node AS n, score

    OPTIONAL MATCH path = (n)-[r*1..2]-(neighbor)
    WHERE NOT neighbor:Document AND ALL(rel IN r WHERE type(rel) <> 'MENTIONS')

    OPTIONAL MATCH (n)-[:MENTIONS]-(doc:Document)

    RETURN n.id AS source, n.description AS source_desc, [rel IN r | type(rel)] AS relations,
           neighbor.id AS target, neighbor.description AS target_desc, labels(neighbor)[0] AS target_type,
           collect(DISTINCT {text: doc.text, link: doc.link}) AS related_news, score
    ORDER BY score DESC LIMIT 15
    """
    response = graph.query(query, params={"embedding": question_vector})
    if not response: return "관련 정보 없음"

    results, seen_entities, processed_news = [], set(), {}
    for row in response:
        if row.get('source') and row['source'] not in seen_entities:
            s_desc = row['source_desc'] or "설명 없음"
            results.append(f"\n[핵심 엔티티] {row['source']} ({s_desc})")
            if row.get('related_news'):
                for news in row['related_news']:
                    if news.get('text'):
                        if news['text'] not in processed_news:
                            processed_news[news['text']] = news_summary_chain.invoke({"text": news['text']})
                        results.append(f"  📰 뉴스 요약: {processed_news[news['text']]}")
                        if news.get('link'): results.append(f"  🔗 출처: {news['link']}")
            seen_entities.add(row['source'])

        if row.get('target'):
            rel_path = " -> ".join(row['relations']) if row.get('relations') else ""
            results.append(f"  - 경로: {row['source']} -[{rel_path}]-> {row['target']} ({row['target_type']})")
            if row['target'] not in seen_entities:
                t_desc = row['target_desc'] or "상세 정보 없음"
                results.append(f"  - 영향 대상: {row['target']} ({t_desc})")
                seen_entities.add(row['target'])

    final_context = "\n".join(results)
    return final_context if final_context.strip() else "관련 정보 없음"

def extract_node_ids(context: str):
    sources = re.findall(r'\[핵심 엔티티\] (.*?) \(', context)
    targets = re.findall(r'- 영향 대상: (.*?) \(', context)
    return list(set(sources + targets))

# --- 3. 대화 기억 장치 추가 및 원본 프롬프트 복구 ---
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 현대자동차 기업의 SCM(공급망 관리) 분석 전문가입니다.
제공된 '참고 정보'는 지식 그래프에서 추출된 엔티티 간의 관계입니다.

이 정보를 바탕으로 질문에 대해 논리적이고 전문적으로 답변하십시오.
만약 정보가 부족하다면 추측하기보다 정보가 부족함을 알리세요.
이전 대화 기록이 있다면 참고하여 맥락을 이어가십시오.

참고 정보:
{context}"""),
    MessagesPlaceholder(variable_name="chat_history"), # 📌 대화 내역 저장소
    ("human", "질문: {question}"),
])
qa_chain = qa_prompt | llm | StrOutputParser()

class QueryRequest(BaseModel):
    question: str
    chat_history: List[Dict[str, str]] = []

@app.post("/query")
async def ask_graph_rag(request: QueryRequest):
    question = request.question

    # 대화 기록 변환
    langchain_history = []
    for msg in request.chat_history:
        if msg["role"] == "user":
            langchain_history.append(HumanMessage(content=msg["content"]))
        elif msg["role"] == "assistant":
            langchain_history.append(AIMessage(content=msg["content"]))

    # 그래프 탐색
    context = graph_retriever(question)
    context_msg = "관련된 SCM 정보가 그래프에 없습니다." if context == "관련 정보 없음" else context

    # 답변 생성
    final_answer = qa_chain.invoke({
        "question": question,
        "context": context_msg,
        "chat_history": langchain_history
    })

    answer_text = final_answer.content if hasattr(final_answer, 'content') else str(final_answer)
    highlight_nodes = extract_node_ids(context)

    return {
        "answer": answer_text.replace('\n', '<br>'),
        "context": context_msg, # 📌 프론트엔드 검증용 컨텍스트 전달
        "nodes_to_highlight": highlight_nodes,
        "used_retriever": "Graph-Cypher Retriever"
    }

@app.get("/graph-data")
async def get_graph():
    return graph.query("MATCH (n)-[r]->(m) RETURN n, r, m LIMIT 100")

# --- 4. 프론트엔드 UI ---
@app.get("/")
async def serve_frontend():
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>GraphRAG Demo</title>
        <script type="text/javascript" src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
        <style>
            body { display: flex; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 0; height: 100vh; }
            #chat-panel { width: 40%; padding: 20px; border-right: 2px solid #ddd; display: flex; flex-direction: column; background: #fdfdfd; box-sizing: border-box; }
            #graph-panel { width: 60%; height: 100%; background: #ffffff; }
            #chat-history { flex-grow: 1; overflow-y: auto; margin-bottom: 15px; border: 1px solid #ccc; border-radius: 8px; padding: 15px; background: white; }
            #mynetwork { width: 100%; height: 100%; outline: none; }
            .message { margin-bottom: 12px; padding: 10px; border-radius: 8px; font-size: 14px; line-height: 1.5; }
            .user { background-color: #e3f2fd; text-align: right; border-bottom-right-radius: 0; }
            .bot { background-color: #f1f8e9; border-bottom-left-radius: 0; border: 1px solid #c5e1a5;}
            input { padding: 10px; border-radius: 5px; border: 1px solid #ccc; width: 100%; margin-bottom: 10px; box-sizing: border-box; }
            button { padding: 10px; border: none; border-radius: 5px; background: #2E86C1; color: white; cursor: pointer; font-weight: bold; width: 100%; box-sizing: border-box;}
            button:hover { background: #21618C; }
            .highlight-info { color: #d35400; font-weight: bold; margin-top: 8px; display: block; font-size: 12px; }
            details { background: #eee; padding: 10px; border-radius: 5px; margin-top: 10px; cursor: pointer; font-size: 13px; }
        </style>
    </head>
    <body>
        <div id="chat-panel">
            <h2 style="color: #2E86C1; margin-top:0;">🤖 SCM 전문가 RAG</h2>
            <div id="chat-history">
                <div class="message bot">DB와 정상 연동되었습니다. 질문을 입력해보세요!</div>
            </div>
            <input type="text" id="question" placeholder="질문을 입력하세요..." onkeypress="if(event.key === 'Enter') askQuestion()">
            <button onclick="askQuestion()">🚀 질문하기</button>
        </div>
        <div id="graph-panel"><div id="mynetwork"></div></div>

        <script>
            let network;
            let nodes = new vis.DataSet();
            let edges = new vis.DataSet();
            let globalChatHistory = []; // 📌 프론트엔드 대화 기록 저장

            async function loadGraph() {
                const response = await fetch('/graph-data');
                const data = await response.json();
                let uniqueNodes = new Map();
                let uniqueEdges = [];
                data.forEach(record => {
                    let n = record.n || {}; let m = record.m || {};
                    let n_id = n.id || n.name || "Unknown1";
                    let m_id = m.id || m.name || "Unknown2";
                    uniqueNodes.set(n_id, { id: n_id, label: n_id, color: '#A9CCE3', shape: 'dot', size: 15 });
                    uniqueNodes.set(m_id, { id: m_id, label: m_id, color: '#A9CCE3', shape: 'dot', size: 15 });
                    uniqueEdges.push({ from: n_id, to: m_id, color: '#D5DBDB' });
                });
                nodes.add(Array.from(uniqueNodes.values()));
                edges.add(uniqueEdges);
                const container = document.getElementById('mynetwork');
                network = new vis.Network(container, { nodes: nodes, edges: edges }, { physics: { stabilization: false } });
            }

            async function askQuestion() {
                const qInput = document.getElementById('question');
                const question = qInput.value;
                if (!question) return;

                const history = document.getElementById('chat-history');
                history.innerHTML += `<div class="message user">${question}</div>`;
                qInput.value = '';
                history.scrollTop = history.scrollHeight;

                try {
                    const loadingId = 'loading-' + Date.now();
                    history.innerHTML += `<div id="${loadingId}" class="message bot">⏳ DB를 탐색하고 분석 중입니다...</div>`;
                    history.scrollTop = history.scrollHeight;

                    const response = await fetch('/query', {
                        method: 'POST',
                        headers: { 'Content-Type': 'application/json' },
                        body: JSON.stringify({ question: question, chat_history: globalChatHistory })
                    });
                    const result = await response.json();

                    globalChatHistory.push({"role": "user", "content": question});
                    globalChatHistory.push({"role": "assistant", "content": result.answer});

                    document.getElementById(loadingId).remove();

                    // 📌 사용자가 직접 DB 검색 내용을 확인할 수 있도록 토글 버튼 추가
                    const contextHtml = `<details><summary>🔍 AI가 참고한 DB 데이터 보기 (클릭)</summary><pre style="white-space:pre-wrap;">${result.context}</pre></details>`;

                    history.innerHTML += `<div class="message bot"><b>분석 결과:</b><br>${result.answer}<br>${contextHtml}<br><span class="highlight-info">🔥 하이라이트: ${result.nodes_to_highlight.join(', ')}</span></div>`;
                    history.scrollTop = history.scrollHeight;

                    let updateArray = [];
                    nodes.forEach(node => {
                        if (result.nodes_to_highlight.includes(node.id)) {
                            updateArray.push({ id: node.id, color: { background: '#E74C3C', border: '#900C3F' }, size: 30, font: {color: 'red', size: 20, bold: true} });
                        } else {
                            updateArray.push({ id: node.id, color: '#A9CCE3', size: 15, font: {color: 'black', size: 14} });
                        }
                    });
                    nodes.update(updateArray);

                    if(result.nodes_to_highlight.length > 0) {
                        network.fit({ nodes: result.nodes_to_highlight, animation: true });
                    }
                } catch(error) {
                    history.innerHTML += `<div class="message bot">서버 오류가 발생했습니다.</div>`;
                }
            }
            window.onload = loadGraph;
        </script>
    </body>
    </html>
    """
    return HTMLResponse(content=html_content)

# --- 5. 서버 실행 ---
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("⏳ 서버 구동 중입니다. 잠시만 기다려주세요...")
time.sleep(2)
print("✅ 구동 완료! 아래에 챗봇 화면이 출력됩니다.")

output.serve_kernel_port_as_iframe(8000, height=750)

INFO:     Started server process [37837]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


⏳ 서버 구동 중입니다. 잠시만 기다려주세요...
✅ 구동 완료! 아래에 챗봇 화면이 출력됩니다.


<IPython.core.display.Javascript object>

In [1]:
import nest_asyncio
import threading
import uvicorn
import time
import re
from typing import List, Dict
from fastapi import FastAPI
from fastapi.responses import HTMLResponse
from pydantic import BaseModel
from google.colab import userdata, output
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# 최신 패키지 임포트
from langchain_neo4j import Neo4jGraph, Neo4jVector

# Colab 비동기 환경 패치
nest_asyncio.apply()
app = FastAPI()

# --- 1. 설정 및 리트리버 준비 ---
NEO4J_URI = ''
NEO4J_USERNAME = ''
NEO4J_PASSWORD = ''
NEO4J_DATABASE = ''
OPENAI_API_KEY = userdata.get('')

graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE)
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
llm = ChatOpenAI(model="gpt-4o", temperature=0, openai_api_key=OPENAI_API_KEY)

vector_index = Neo4jVector.from_existing_index(
    embeddings, url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD, database=NEO4J_DATABASE,
    index_name="entity_vector_index", text_node_property="id"
)

# --- 2. SCM 분석 및 뉴스 요약 로직 ---
news_summary_prompt = ChatPromptTemplate.from_template(
    "다음 뉴스 원문을 SCM(공급망) 리스크 관점에서 1~2문장으로 핵심만 요약해줘. 관련 링크가 있다면 포함해.\n\n원문: {text}"
)
news_summary_chain = news_summary_prompt | llm | StrOutputParser()

def graph_retriever(question: str) -> str:
    question_vector = embeddings.embed_query(question)

    # 📌 [핵심 수정] 단일 노드 독식 방지! 각 노드별로 관계는 4개, 뉴스는 2개까지만 제한해서 다양성 확보
    query = """
    CALL db.index.vector.queryNodes('entity_vector_index', 4, $embedding)
    YIELD node AS n, score

    CALL {
        WITH n
        OPTIONAL MATCH (n)-[r*1..2]-(neighbor)
        WHERE NOT neighbor:Document AND ALL(rel IN r WHERE type(rel) <> 'MENTIONS')
        RETURN neighbor, r
        LIMIT 4
    }

    CALL {
        WITH n
        OPTIONAL MATCH (n)-[:MENTIONS]-(doc:Document)
        RETURN doc
        LIMIT 2
    }

    RETURN n.id AS source, n.description AS source_desc,
           [rel IN r | type(rel)] AS relations,
           neighbor.id AS target, neighbor.description AS target_desc, labels(neighbor)[0] AS target_type,
           collect(DISTINCT {text: substring(doc.text, 0, 1000), link: doc.link}) AS related_news,
           score
    ORDER BY score DESC
    """

    response = graph.query(query, params={"embedding": question_vector})
    if not response: return "관련 정보 없음"

    results, seen_entities, processed_news = [], set(), {}
    for row in response:
        if row.get('source') and row['source'] not in seen_entities:
            s_desc = row['source_desc'] or "설명 없음"
            results.append(f"\n[핵심 엔티티] {row['source']} ({s_desc})")

            if row.get('related_news'):
                for news in row['related_news']:
                    if news.get('text'):
                        text_snippet = news['text']
                        if text_snippet not in processed_news:
                            try:
                                # 📌 [핵심 수정] 서버 과부하 방지 및 예외처리
                                processed_news[text_snippet] = news_summary_chain.invoke({"text": text_snippet})
                            except Exception as e:
                                processed_news[text_snippet] = "뉴스 요약 중 일시적 오류 발생"

                        results.append(f"  📰 뉴스 요약: {processed_news[text_snippet]}")
                        if news.get('link'): results.append(f"  🔗 출처: {news['link']}")
            seen_entities.add(row['source'])

        if row.get('target'):
            rel_path = " -> ".join(row['relations']) if row.get('relations') else ""
            results.append(f"  - 경로: {row['source']} -[{rel_path}]-> {row['target']} ({row['target_type']})")
            if row['target'] not in seen_entities:
                t_desc = row['target_desc'] or "상세 정보 없음"
                results.append(f"  - 영향 대상: {row['target']} ({t_desc})")
                seen_entities.add(row['target'])

    final_context = "\n".join(results)
    return final_context if final_context.strip() else "관련 정보 없음"

def extract_node_ids(context: str):
    sources = re.findall(r'\[핵심 엔티티\] (.*?) \(', context)
    targets = re.findall(r'- 영향 대상: (.*?) \(', context)
    return list(set(sources + targets))

# --- 3. 대화 기억 장치 추가된 프롬프트 및 API ---
qa_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 현대자동차 기업의 SCM(공급망 관리) 분석 전문가입니다.
반드시 아래 제공된 '참고 정보'의 내용만을 바탕으로 답변하십시오.

[엄격한 제약 조건]
1. 일반적인 지식이나 추측을 섞어 대답하지 마십시오. 오직 참고 정보에 있는 엔티티, 관계, 뉴스 요약에 대해서만 비교 분석하여 답변하세요.
2. 질문과 관련된 구체적인 정보가 없다면 "현재 지식 그래프 데이터상으로는 해당 내용을 확인할 수 없습니다"라고 명확히 답하십시오.
3. 이전 대화 기록을 기억하여, 사용자가 이전 답변과 연결해서 질문할 경우 맥락에 맞게 답변하십시오.

참고 정보:
{context}"""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "질문: {question}"),
])
qa_chain = qa_prompt | llm | StrOutputParser()

class QueryRequest(BaseModel):
    question: str
    chat_history: List[Dict[str, str]] = []

@app.post("/query")
async def ask_graph_rag(request: QueryRequest):
    question = request.question

    # 대화 기록 변환
    langchain_history = []
    for msg in request.chat_history:
        if msg["role"] == "user":
            langchain_history.append(HumanMessage(content=msg["content"]))
        elif msg["role"] == "assistant":
            langchain_history.append(AIMessage(content=msg["content"]))

    # 그래프 탐색
    context = graph_retriever(question)
    context_msg = "관련된 SCM 정보가 그래프에 없습니다." if context == "관련 정보 없음" else context

    try:
        # AI 전문가 분석 답변 생성
        final_answer = qa_chain.invoke({
            "question": question,
            "context": context_msg,
            "chat_history": langchain_history
        })
        answer_text = final_answer.content if hasattr(final_answer, 'content') else str(final_answer)
    except Exception as e:
        answer_text = f"답변 생성 중 오류가 발생했습니다: {str(e)}"

    highlight_nodes = extract_node_ids(context)

    return {
        "answer": answer_text.replace('\n', '<br>'),
        "context": context_msg,
        "nodes_to_highlight": highlight_nodes,
        "used_retriever": "Graph-Cypher Retriever"
    }

@app.get("/graph-data")
async def get_graph():
    return graph.query("MATCH (n)-[r]->(m) RETURN n, r, m LIMIT 100")

# --- 4. 프론트엔드 UI ---
@app.get("/")
async def serve_frontend():
    html_content = """
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="utf-8">
        <title>GraphRAG Demo</title>
        <script type="text/javascript" src="https://unpkg.com/vis-network/standalone/umd/vis-network.min.js"></script>
        <style>
            body { display: flex; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 0; height: 100vh; }
            #chat-panel { width: 40%; padding: 20px; border-right: 2px solid #ddd; display: flex; flex-direction: column; background: #fdfdfd; box-sizing: border-box; }
            #graph-panel { width: 60%; height: 100%; background: #ffffff; }
            #chat-history { flex-grow: 1; overflow-y: auto; margin-bottom: 15px; border: 1px solid #ccc; border-radius: 8px; padding: 15px; background: white; }
            #mynetwork { width: 100%; height: 100%; outline: none; }
            .message { margin-bottom: 12px; padding: 10px; border-radius: 8px; font-size: 14px; line-height: 1.5; }
            .user { background-color: #e3f2fd; text-align: right; border-bottom-right-radius: 0; }
            .bot { background-color: #f1f8e9; border-bottom-left-radius: 0; border: 1px solid #c5e1a5;}
            input { padding: 10px; border-radius: 5px; border: 1px solid #ccc; width: 100%; margin-bottom: 10px; box-sizing: border-box; }
            button { padding: 10px; border: none; border-radius: 5px; background: #2E86C1; color: white; cursor: pointer; font-weight: bold; width: 100%; box-sizing: border-box;}
            button:hover { background: #21618C; }
            .highlight-info { color: #d35400; font-weight: bold; margin-top: 8px; display: block; font-size: 12px; }
            details { background: #eee; padding: 10px; border-radius: 5px; margin-top: 10px; cursor: pointer; font-size: 13px; border-left: 3px solid #2E86C1;}
        </style>
    </head>
    <body>
        <div id="chat-panel">
            <h2 style="color: #2E86C1; margin-top:0;">🤖 SCM 전문가 RAG (업그레이드)</h2>
            <div id="chat-history">
                <div class="message bot">✅ DB 탐색 다양성 확보 & 서버 에러 방지 완료!<br>여러 공급처를 비교 분석할 수 있습니다. 질문해보세요!</div>
            </div>
            <input type="text" id="question" placeholder="질문을 입력하세요..." onkeypress="if(event.key === 'Enter') askQuestion()">
            <button onclick="askQuestion()">🚀 질문하기</button>
        </div>
        <div id="graph-panel"><div id="mynetwork"></div></div>

        <script>
            let network;
            let nodes = new vis.DataSet();
            let edges = new vis.DataSet();
            let globalChatHistory = [];

            async function loadGraph() {
                const response = await fetch('/graph-data');
                const data = await response.json();
                let uniqueNodes = new Map();
                let uniqueEdges = [];
                data.forEach(record => {
                    let n = record.n || {}; let m = record.m || {};
                    let n_id = n.id || n.name || "Unknown1";
                    let m_id = m.id || m.name || "Unknown2";
                    uniqueNodes.set(n_id, { id: n_id, label: n_id, color: '#A9CCE3', shape: 'dot', size: 15 });
                    uniqueNodes.set(m_id, { id: m_id, label: m_id, color: '#A9CCE3', shape: 'dot', size: 15 });
                    uniqueEdges.push({ from: n_id, to: m_id, color: '#D5DBDB' });
                });
                nodes.add(Array.from(uniqueNodes.values()));
                edges.add(uniqueEdges);
                const container = document.getElementById('mynetwork');
                network = new vis.Network(container, { nodes: nodes, edges: edges }, { physics: { stabilization: false } });
            }

            async function askQuestion() {
                const qInput = document.getElementById('question');
                const question = qInput.value;
                if (!question) return;

                const history = document.getElementById('chat-history');
                history.innerHTML += `<div class="message user">${question}</div>`;
                qInput.value = '';
                history.scrollTop = history.scrollHeight;

                try {
                    const loadingId = 'loading-' + Date.now();
                    history.innerHTML += `<div id="${loadingId}" class="message bot">⏳ 여러 엔티티를 탐색하고 뉴스를 요약 중입니다...</div>`;
                    history.scrollTop = history.scrollHeight;

                    const response = await fetch('/query', {
                        method: 'POST',
                        headers: { 'Content-Type': 'application/json' },
                        body: JSON.stringify({ question: question, chat_history: globalChatHistory })
                    });
                    const result = await response.json();

                    globalChatHistory.push({"role": "user", "content": question});
                    globalChatHistory.push({"role": "assistant", "content": result.answer});

                    document.getElementById(loadingId).remove();

                    const contextHtml = `<details><summary>🔍 AI가 수집한 DB 데이터 보기 (다양한 기업 확인)</summary><pre style="white-space:pre-wrap; font-family:inherit;">${result.context}</pre></details>`;

                    history.innerHTML += `<div class="message bot"><b>분석 결과:</b><br>${result.answer}<br>${contextHtml}<br><span class="highlight-info">🔥 하이라이트: ${result.nodes_to_highlight.join(', ')}</span></div>`;
                    history.scrollTop = history.scrollHeight;

                    let updateArray = [];
                    nodes.forEach(node => {
                        if (result.nodes_to_highlight.includes(node.id)) {
                            updateArray.push({ id: node.id, color: { background: '#E74C3C', border: '#900C3F' }, size: 30, font: {color: 'red', size: 20, bold: true} });
                        } else {
                            updateArray.push({ id: node.id, color: '#A9CCE3', size: 15, font: {color: 'black', size: 14} });
                        }
                    });
                    nodes.update(updateArray);

                    if(result.nodes_to_highlight.length > 0) {
                        network.fit({ nodes: result.nodes_to_highlight, animation: true });
                    }
                } catch(error) {
                    history.innerHTML += `<div class="message bot">서버와 통신 중 오류가 발생했습니다. 잠시 후 다시 시도해주세요.</div>`;
                }
            }
            window.onload = loadGraph;
        </script>
    </body>
    </html>
    """
    return HTMLResponse(content=html_content)

# --- 5. 서버 실행 ---
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("⏳ 서버 구동 중입니다. 잠시만 기다려주세요...")
time.sleep(2)
print("✅ 구동 완료! 아래에 챗봇 화면이 출력됩니다.")

output.serve_kernel_port_as_iframe(8000, height=750)

INFO:     Started server process [39758]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


⏳ 서버 구동 중입니다. 잠시만 기다려주세요...
✅ 구동 완료! 아래에 챗봇 화면이 출력됩니다.


<IPython.core.display.Javascript object>